# Spiral world map for 3D printing (Bambu H2C, 2 colours)

A world map printed on a spiral strip that hangs from the ceiling. Seen from below, all turns together form one map with the north pole in the centre. Two shapes share the same map code.

| `MODE` | Shape | How the map is printed | Pieces |
|---|---|---|---|
| `"flat"` | Flat disc cut into a spiral. Hung from the centre, the turns drop into a cone. | Continents as **raised relief** on top of a clear strip | Square plates on a grid, plus glue-on splices |
| `"cone"` | Strip already shaped as a 3D cone, wide at the top. | Map **inlaid** in the underside | Angular pieces along the spiral, plus glue-on splices |

**Loading in Bambu Studio.** Drag the `*_clear.stl` and `*_map.stl` of one plate or piece in together and answer *Yes* to "load as a single object with multiple parts". Then give the `map` part the colour filament.

**Map source.** By default the land polygons of Natural Earth (`ne_50m_land.geojson`). Set `DATA_IMAGE` to any greyscale equirectangular world map (for example light or ocean pollution) to print that instead.

**Pipeline.** Settings → world map lookup → STL writer → flat geometry → cone geometry → selector.

In [ ]:
# Cell 1: settings (shared ones first, then Flat and Cone)

import json
import math
import os

import numpy as np
from PIL import Image, ImageDraw, ImageFilter
import matplotlib.pyplot as plt

MODE = "flat"             # "flat" = flat disc with relief, "cone" = 3D cone with inlay

# ---------------------------------------------------------------- map (shared)
LAND_GEOJSON = "../data/ne_50m_land.geojson"
DATA_IMAGE = None         # optional greyscale lon/lat world map, e.g. pollution data
THRESHOLD = 0.5           # DATA_IMAGE only, brighter pixels get the map colour
INVERT = False            # True = colour the oceans instead of land
SOUTH_LIMIT = 140         # degrees from the pole at the outer edge (140 = 50°S)
ROTATE_DEG = 0

# ---------------------------------------------------------------- printer (shared)
BED = (325, 320, 325)     # build volume, check your printer
RES = 1.0                 # mesh cell in mm, smaller = sharper and bigger files


class Flat:
    """Settings for the flat disc spiral (MODE = "flat")."""
    DIAMETER = 600        # 600 = 2 x 2 plates
    TURNS = 6             # more turns = narrower strip, smoother drop
    HUB_R = 40            # solid centre disc the spiral starts from
    GAP = 1.5             # cut between neighbouring turns
    LAYER = 0.3           # must match the slicer, heights are rounded to whole layers
    THICKNESS = 1.2       # keep it thin (0.8 to 1.6) so the strip flexes
    RELIEF = 0.9          # height of the raised map
    RELIEF_LEVELS = 1     # >1 turns DATA_IMAGE values into height steps
    HANG_HOLE = 6         # centre hole for the ceiling hook
    TIP_HOLE = 4          # hole at the outer end for a weight (0 = off)
    STRING_HOLES = 0      # holes per turn for fishing line (0 = off)
    STRING_HOLE = 2.0
    BED_MARGIN = 5
    SPLICE_LEN = 24       # splice length across the joint
    SPLICE_T = 0.9
    OUT_DIR = "flat_spiral_parts"


class Cone:
    """Settings for the conical spiral (MODE = "cone")."""
    R_OUT = 150           # outer radius at the top
    R_IN = 20             # inner radius at the bottom
    HEIGHT = 600          # total drop
    TURNS = 3
    THICKNESS = 3.0
    INLAY = 0.6           # depth of the map colour on the underside
    PIECES = 4
    SPLICE_DEG = 12       # splice length along the strip, in degrees
    SPLICE_T = 1.6
    OUT_DIR = "spiral_parts"

## World map lookup
The map is rasterised once into a lon/lat grid. `world_at` reads it at any lon/lat, either smooth (bilinear, for the flat relief edges) or crisp (nearest pixel, for the cone inlay cells).

In [ ]:
# Cell 2: world map lookup

def load_world(res_deg=0.1):
    """Rasterise the map source into a lon/lat grid with values 0 to 1."""
    width, height = int(360 / res_deg), int(180 / res_deg)
    if DATA_IMAGE:
        img = Image.open(DATA_IMAGE).convert("L").resize((width, height))
    else:
        img = Image.new("L", (width, height), 0)
        draw = ImageDraw.Draw(img)
        with open(LAND_GEOJSON) as f:
            features = json.load(f)["features"]
        for feature in features:
            geometry = feature["geometry"]
            if geometry["type"] == "MultiPolygon":
                polygons = geometry["coordinates"]
            else:
                polygons = [geometry["coordinates"]]
            for polygon in polygons:
                for ring_index, ring in enumerate(polygon):
                    ring = np.asarray(ring)
                    xs = (ring[:, 0] + 180) / res_deg
                    ys = (90 - ring[:, 1]) / res_deg
                    fill = 255 if ring_index == 0 else 0      # ring 0 = coastline, others = lakes
                    draw.polygon(list(zip(xs, ys)), fill=fill)

    crisp = np.asarray(img, dtype=np.float32) / 255
    smooth = np.asarray(img.filter(ImageFilter.GaussianBlur(1)), dtype=np.float32) / 255
    return crisp, smooth


WORLD_CRISP, WORLD_SMOOTH = load_world()   # crisp for the cone, smooth for the flat disc


def world_at(lon, lat, smooth=True):
    """Read the map at given coordinates."""
    if not smooth:
        h, w = WORLD_CRISP.shape
        col = ((lon + 180) % 360 / 360 * w).astype(int) % w
        row = np.clip(((90 - lat) / 180 * h).astype(int), 0, h - 1)
        return WORLD_CRISP[row, col]

    h, w = WORLD_SMOOTH.shape
    fx = (lon + 180) % 360 / 360 * w - 0.5
    fy = np.clip((90 - lat) / 180 * h - 0.5, 0, h - 1.001)
    x0, y0 = np.floor(fx).astype(int), np.floor(fy).astype(int)
    tx, ty = fx - x0, fy - y0
    x0, x1 = x0 % w, (x0 + 1) % w
    top = WORLD_SMOOTH[y0, x0] * (1 - tx) + WORLD_SMOOTH[y0, x1] * tx
    bottom = WORLD_SMOOTH[y0 + 1, x0] * (1 - tx) + WORLD_SMOOTH[y0 + 1, x1] * tx
    return top * (1 - ty) + bottom * ty


def map_latlon(r, angle, r_max, direction):
    """Polar projection used by both shapes. Centre = north pole, r_max = SOUTH_LIMIT."""
    lat = 90 - r / r_max * SOUTH_LIMIT
    lon = ROTATE_DEG + direction * np.degrees(angle)
    return lon, lat

## STL writer
Both shapes produce plain triangle arrays, written as binary STL.

In [ ]:
# Cell 3: STL writer

def write_stl(path, tris, header="spiral map"):
    """Write triangles to a binary STL file."""
    tris = tris.astype(np.float32)
    normals = np.cross(tris[:, 1] - tris[:, 0], tris[:, 2] - tris[:, 0])
    normals /= np.linalg.norm(normals, axis=1, keepdims=True) + 1e-12
    record = np.zeros(len(tris), dtype=[("n", "<f4", 3), ("v", "<f4", (3, 3)), ("a", "<u2")])
    record["n"], record["v"] = normals, tris
    with open(path, "wb") as f:
        f.write(header.encode().ljust(80, b" "))
        f.write(np.uint32(len(tris)).tobytes())
        f.write(record.tobytes())


def clear_stl_folder(folder):
    """Create the folder and delete STL files left from a previous run."""
    os.makedirs(folder, exist_ok=True)
    for name in os.listdir(folder):
        if name.endswith(".stl"):
            os.remove(os.path.join(folder, name))

## Flat disc: geometry as fields
Every part is described by a **field** that is positive inside the part and negative outside, roughly the distance to its edge. Combining parts is then just `min` (intersection). The strip field looks at each turn of the spiral and keeps the closest one.

In [ ]:
# Cell 4: flat disc geometry (fields, > 0 inside)

def flat_dims(turns, diameter):
    """Basic sizes of the flat spiral."""
    R = diameter / 2
    pitch = (R - Flat.HUB_R) / turns
    width = pitch - Flat.GAP
    theta_max = 2 * np.pi * (R - Flat.HUB_R - width) / pitch     # outer edge of the last turn = R
    return R, pitch, width, theta_max


def flat_centreline(turns, diameter, theta):
    """Point on the middle line of the strip."""
    _, pitch, width, _ = flat_dims(turns, diameter)
    rc = Flat.HUB_R + width / 2 + pitch * theta / (2 * np.pi)
    return rc * np.cos(theta), rc * np.sin(theta)


def strip_field(x, y, turns, diameter, only_turn=None):
    """Field of the spiral strip and the centre hub."""
    _, pitch, width, theta_max = flat_dims(turns, diameter)
    r = np.hypot(x, y)
    phi = np.arctan2(y, x) % (2 * np.pi)
    best = np.full(r.shape, -1e9, dtype=np.float64)
    turn = np.full(r.shape, -1, dtype=int)

    for k in range(turns + 1):
        if only_turn is not None and k != only_turn:
            continue
        theta = phi + 2 * np.pi * k
        rc = Flat.HUB_R + width / 2 + pitch * theta / (2 * np.pi)
        across = width / 2 - np.abs(r - rc)            # distance to the side edges
        end_cap = (theta_max - theta) * rc              # square end of the spiral
        d = np.minimum(across, end_cap)
        closer = d > best
        best[closer], turn[closer] = d[closer], k

    if only_turn is None or only_turn == -1:
        # The hub grows with the spiral so it keeps a GAP to turn 0.
        hub = Flat.HUB_R + pitch * phi / (2 * np.pi) - Flat.GAP - r
        if only_turn == -1:
            best = hub
        else:
            closer = hub > best
            best[closer], turn[closer] = hub[closer], -1
    return best, turn


def flat_holes(turns, diameter):
    """Positions of all holes."""
    R, _, width, theta_max = flat_dims(turns, diameter)
    holes = [(0.0, 0.0, Flat.HANG_HOLE / 2)]
    if Flat.TIP_HOLE:
        rc_end = R - width / 2
        theta = theta_max - (width * 0.8) / rc_end
        holes.append((*flat_centreline(turns, diameter, theta), Flat.TIP_HOLE / 2))
    if Flat.STRING_HOLES:
        for m in range(1, int(theta_max / (2 * np.pi) * Flat.STRING_HOLES)):
            theta = 2 * np.pi * m / Flat.STRING_HOLES
            if theta < theta_max - 0.3:
                holes.append((*flat_centreline(turns, diameter, theta), Flat.STRING_HOLE / 2))
    return holes


def flat_part_fields(x, y, turns, diameter):
    """Fields of the clear strip and of each relief level."""
    strip, _ = strip_field(x, y, turns, diameter)
    for hx, hy, hr in flat_holes(turns, diameter):
        strip = np.minimum(strip, np.hypot(x - hx, y - hy) - hr)

    lon, lat = map_latlon(np.hypot(x, y), np.arctan2(y, x), diameter / 2, +1)
    value = world_at(lon, lat, smooth=True)
    if INVERT:
        value = 1 - value

    start = THRESHOLD if DATA_IMAGE else 0.5
    steps = [start + (1 - start) * k / Flat.RELIEF_LEVELS for k in range(Flat.RELIEF_LEVELS)]
    levels = [np.minimum(strip - 0.3, (value - t) * 4) for t in steps]
    return strip, levels

## Flat disc: field to mesh
A filled **marching squares** walk turns each field into a 2D outline, which is extruded into a closed slab. Edge points are interpolated, so a 1 mm grid already gives smooth edges.

In [ ]:
# Cell 5: flat disc meshing (filled marching squares, extruded)

def slab_mesh(F, X, Y, z0, z1):
    """Closed, watertight triangles of the region F > 0, extruded from z0 to z1."""
    inside = F > 0

    def crossing(fa, fb, pa, pb):
        # Where the field crosses 0 along a grid edge. Computed once per edge,
        # so neighbouring cells share exactly the same points.
        with np.errstate(divide="ignore", invalid="ignore"):
            t = np.where(fa != fb, fa / (fa - fb), 0.5)
        return pa + (pb - pa) * np.clip(t, 0, 1)

    hx = crossing(F[:-1, :], F[1:, :], X[:-1, :], X[1:, :])
    hy = crossing(F[:-1, :], F[1:, :], Y[:-1, :], Y[1:, :])
    vx = crossing(F[:, :-1], F[:, 1:], X[:, :-1], X[:, 1:])
    vy = crossing(F[:, :-1], F[:, 1:], Y[:, :-1], Y[:, 1:])

    P = np.stack([
        np.stack([X[:-1, :-1], Y[:-1, :-1]], -1),   # c0
        np.stack([hx[:, :-1], hy[:, :-1]], -1),     # e0 bottom edge
        np.stack([X[1:, :-1], Y[1:, :-1]], -1),     # c1
        np.stack([vx[1:, :], vy[1:, :]], -1),       # e1 right edge
        np.stack([X[1:, 1:], Y[1:, 1:]], -1),       # c2
        np.stack([hx[:, 1:], hy[:, 1:]], -1),       # e2 top edge
        np.stack([X[:-1, 1:], Y[:-1, 1:]], -1),     # c3
        np.stack([vx[:-1, :], vy[:-1, :]], -1),     # e3 left edge
    ], axis=2)                                      # (nx, ny, 8, 2)

    corner_in = np.stack([inside[:-1, :-1], inside[1:, :-1], inside[1:, 1:], inside[:-1, 1:]], -1)
    valid = np.zeros(corner_in.shape[:2] + (8,), bool)
    valid[..., 0::2] = corner_in
    for e in range(4):
        valid[..., 2 * e + 1] = corner_in[..., e] != corner_in[..., (e + 1) % 4]

    keep = valid.any(-1)
    P, valid, corner_in = P[keep], valid[keep], corner_in[keep]
    centre = F[:-1, :-1][keep] + F[1:, :-1][keep] + F[1:, 1:][keep] + F[:-1, 1:][keep]

    # Saddles with an outside centre: two separate corner triangles.
    c = corner_in
    saddle_a = c[:, 0] & c[:, 2] & ~c[:, 1] & ~c[:, 3] & (centre <= 0)
    saddle_b = c[:, 1] & c[:, 3] & ~c[:, 0] & ~c[:, 2] & (centre <= 0)
    polys = [(P, valid & ~(saddle_a | saddle_b)[:, None])]
    for selected, slots1, slots2 in ((saddle_a, [0, 1, 7], [3, 4, 5]),
                                     (saddle_b, [1, 2, 3], [5, 6, 7])):
        for slots in (slots1, slots2):
            v = np.zeros((selected.sum(), 8), bool)
            v[:, slots] = True
            polys.append((P[selected], v))

    def at_z(points, z):
        return np.column_stack([points, np.full(len(points), z)])

    tris = []
    for Pp, v in polys:
        if len(Pp) == 0:
            continue
        order = np.argsort(~v, axis=1, kind="stable")    # valid slots first, walk order kept
        Q = np.take_along_axis(Pp, order[..., None], 1)
        count = v.sum(1)

        # Top and bottom faces: fan from the first point.
        for k in range(1, 7):
            m = count >= k + 2
            if not m.any():
                continue
            a, b, cc = Q[m, 0], Q[m, k], Q[m, k + 1]
            tris.append(np.stack([at_z(a, z1), at_z(b, z1), at_z(cc, z1)], 1))   # top, CCW
            tris.append(np.stack([at_z(a, z0), at_z(cc, z0), at_z(b, z0)], 1))   # bottom, reversed

        # Walls: between two consecutive crossing points (odd slots).
        for k in range(6):
            m = count > k
            if not m.any():
                continue
            nxt = np.where(k + 1 < count, k + 1, 0)
            i = np.nonzero(m)[0]
            slot_a, slot_b = order[i, k], order[i, nxt[i]]
            is_wall = (slot_a % 2 == 1) & (slot_b % 2 == 1)
            if not is_wall.any():
                continue
            i = i[is_wall]
            pa, pb = Q[i, k], Q[i, nxt[i]]
            A0, B0, A1, B1 = at_z(pa, z0), at_z(pb, z0), at_z(pa, z1), at_z(pb, z1)
            tris.append(np.stack([A0, B0, B1], 1))
            tris.append(np.stack([A0, B1, A1], 1))

    return np.concatenate(tris) if tris else np.zeros((0, 3, 3))


def node_grid(x0, x1, y0, y1):
    """Grid nodes covering a box, offset by half a cell."""
    xs = np.arange(math.floor(x0 / RES) - 1, math.ceil(x1 / RES) + 2) * RES + RES / 2
    ys = np.arange(math.floor(y0 / RES) - 1, math.ceil(y1 / RES) + 2) * RES + RES / 2
    return np.meshgrid(xs, ys, indexing="ij")

## Flat disc: tiles, splices and preview
The disc is cut into a square grid of plates that fit the bed. Wherever the strip crosses a plate border, a small splice plate is generated to glue on the flat side.

In [ ]:
# Cell 6: flat disc generation

def flat_layer_heights():
    """Strip and relief heights, rounded to whole layers."""
    t_strip = max(1, round(Flat.THICKNESS / Flat.LAYER)) * Flat.LAYER
    heights = [max(1, round(Flat.RELIEF * k / Flat.RELIEF_LEVELS / Flat.LAYER)) * Flat.LAYER
               for k in range(1, Flat.RELIEF_LEVELS + 1)]
    return t_strip, heights


def flat_write_plates(turns, diameter, edges):
    """Write the clear and map STL of every plate in the grid."""
    t_strip, heights = flat_layer_heights()
    n = len(edges) - 1
    plate = 0
    for i in range(n):
        for j in range(n):
            x0, x1, y0, y1 = edges[i], edges[i + 1], edges[j], edges[j + 1]
            X, Y = node_grid(x0, x1, y0, y1)
            clip = np.minimum.reduce([X - x0, x1 - X, Y - y0, y1 - Y])   # field of the plate box
            strip, levels = flat_part_fields(X, Y, turns, diameter)
            strip = np.minimum(strip, clip)
            if not (strip > 0).any():
                continue
            plate += 1
            shift = np.array([-(x0 + x1) / 2, -(y0 + y1) / 2, 0])        # centre the plate on the bed
            name = f"{Flat.OUT_DIR}/plate_{plate:02d}_r{j + 1}c{i + 1}"
            write_stl(f"{name}_clear.stl", slab_mesh(strip, X, Y, 0, t_strip) + shift)

            # One slab per relief level, each starting on top of the strip.
            relief = [slab_mesh(np.minimum(f, clip), X, Y, t_strip, t_strip + h)
                      for f, h in zip(levels, heights)]
            relief = [m for m in relief if len(m)]
            if relief:
                write_stl(f"{name}_map.stl", np.concatenate(relief) + shift)
    return plate


def flat_splices(turns, diameter, edges):
    """One splice mesh for every place the strip crosses a plate border."""
    R = diameter / 2
    L = Flat.SPLICE_LEN
    splices = []
    for axis in (0, 1):                       # 0 = vertical borders, 1 = horizontal borders
        for e in edges[1:-1]:
            t = np.arange(-R, R, 0.25)
            px, py = (np.full_like(t, e), t) if axis == 0 else (t, np.full_like(t, e))
            f, turn = strip_field(px, py, turns, diameter)
            inside = f > 0
            starts = np.nonzero(inside & ~np.r_[False, inside[:-1]])[0]
            ends = np.nonzero(inside & ~np.r_[inside[1:], False])[0]

            for a, b in zip(starts, ends):
                k = turn[(a + b) // 2]
                lo, hi = t[a] - L, t[b] + L
                if axis == 0:
                    box = (e - L / 2, e + L / 2, lo, hi)
                else:
                    box = (lo, hi, e - L / 2, e + L / 2)
                X, Y = node_grid(*box)
                across = X if axis == 0 else Y
                along = Y if axis == 0 else X
                band = L / 2 - np.abs(across - e)
                sf, _ = strip_field(X, Y, turns, diameter, only_turn=k)
                sf = np.minimum.reduce([sf - 1.0, band, along - lo, hi - along])
                for hx, hy, hr in flat_holes(turns, diameter):
                    sf = np.minimum(sf, np.hypot(X - hx, Y - hy) - hr - 1)
                mesh = slab_mesh(sf, X, Y, 0, Flat.SPLICE_T)
                if len(mesh):
                    splices.append(mesh)
    return splices


def pack_splices(splices, usable):
    """Lay splices out in rows and write them to as few bed-sized sheets as needed."""
    sheet, x, y, row_h, sheets = [], 0.0, 0.0, 0.0, 0
    for mesh in splices:
        v = mesh.reshape(-1, 3)
        lo, size = v.min(0), np.ptp(v, 0)
        if x + size[0] > usable:              # next row
            x, y, row_h = 0.0, y + row_h + 3, 0.0
        if y + size[1] > usable:              # next sheet
            if sheet:
                sheets += 1
                write_stl(f"{Flat.OUT_DIR}/splices_{sheets}.stl", np.concatenate(sheet))
            sheet, x, y, row_h = [], 0.0, 0.0, 0.0
        sheet.append(mesh - lo + [x - usable / 2, y - usable / 2, 0])
        x += size[0] + 3
        row_h = max(row_h, size[1])
    if sheet:
        sheets += 1
        write_stl(f"{Flat.OUT_DIR}/splices_{sheets}.stl", np.concatenate(sheet))
    return sheets


def flat_preview(turns, diameter, edges):
    """Plot the relief side with plate borders (red) and holes (orange)."""
    R = diameter / 2
    X, Y = np.meshgrid(np.linspace(-R, R, 900), np.linspace(-R, R, 900), indexing="ij")
    strip, levels = flat_part_fields(X, Y, turns, diameter)
    img = np.ones(X.shape + (3,))
    img[strip > 0] = (0.80, 0.88, 0.95)                    # clear strip
    for k, f in enumerate(levels):                         # relief, darker = higher
        img[f > 0] = np.array((0.35, 0.45, 0.65)) * (1 - 0.6 * (k + 1) / len(levels))

    fig, ax = plt.subplots(figsize=(9, 9))
    ax.imshow(img.transpose(1, 0, 2), origin="lower", extent=(-R, R, -R, R))
    for e in edges:
        ax.axvline(e, color="red", lw=0.8)
        ax.axhline(e, color="red", lw=0.8)
    for hx, hy, hr in flat_holes(turns, diameter):
        ax.add_patch(plt.Circle((hx, hy), max(hr, R / 150), color="orange"))
    ax.set_xlim(-R, R)
    ax.set_ylim(-R, R)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title("relief side, as printed and as seen from below when hung")
    plt.show()


def generate_flat(turns=Flat.TURNS, diameter=Flat.DIAMETER):
    """Build all plates, splices and the preview for the flat disc."""
    R, _, width, theta_max = flat_dims(turns, diameter)
    if width < 8:
        print(f"!! strip only {width:.1f} mm wide, use fewer turns or a bigger diameter")
        return

    usable = min(BED[:2]) - 2 * Flat.BED_MARGIN
    n = math.ceil(diameter / usable)
    edges = np.linspace(-R, R, n + 1)
    clear_stl_folder(Flat.OUT_DIR)

    line = np.array(flat_centreline(turns, diameter, np.linspace(0, theta_max, 4000)))
    length = np.hypot(*np.diff(line, axis=1)).sum()
    print(f"{diameter} mm disc, {turns} turns: strip {width:.1f} mm wide, "
          f"{length / 1000:.1f} m long; {n} x {n} grid of {diameter / n:.0f} mm plates")

    plates = flat_write_plates(turns, diameter, edges)
    splices = flat_splices(turns, diameter, edges)
    sheets = pack_splices(splices, usable)
    print(f"{plates} strip plates + {sheets} splice plate(s) ({len(splices)} splices) -> {Flat.OUT_DIR}/")
    flat_preview(turns, diameter, edges)

## Cone: geometry on the strip surface
The cone strip is described in its own coordinates: angle `th` along the spiral, `frac` across the strip (0 = outer edge, 1 = inner edge) and height `dz` above the underside. A small voxel grid in those coordinates is meshed by keeping only the faces between filled and empty cells.

In [ ]:
# Cell 7: cone geometry

def cone_dims(turns):
    """Basic sizes of the conical spiral."""
    width = (Cone.R_OUT - Cone.R_IN) / turns
    return width, 2 * np.pi * turns


def cone_points(turns, th, frac, dz):
    """3D points on the strip."""
    width, theta_total = cone_dims(turns)
    r = Cone.R_OUT - width * th / (2 * np.pi) - frac * width
    z = Cone.HEIGHT * (1 - th / theta_total) + dz
    return np.stack([r * np.cos(th), r * np.sin(th), z], axis=-1)


def voxel_mesh(P, occ):
    """Closed surface of the occupied cells of a curved voxel grid."""
    # Cell centres = mean of the 8 corners.
    C = sum(P[i:i + P.shape[0] - 1, j:j + P.shape[1] - 1, k:k + P.shape[2] - 1]
            for i in (0, 1) for j in (0, 1) for k in (0, 1)) / 8
    tris = []
    for a in range(3):
        pad = [(0, 0)] * 3
        pad[a] = (1, 1)
        o = np.pad(occ, pad)
        lo = np.take(o, range(o.shape[a] - 1), axis=a)
        hi = np.take(o, range(1, o.shape[a]), axis=a)
        face = lo != hi
        idx = np.argwhere(face)               # plane index along a, cell index along the others
        if len(idx) == 0:
            continue
        b, c = [x for x in range(3) if x != a]

        def corner(db, dc):
            q = idx.copy()
            q[:, b] += db
            q[:, c] += dc
            return P[q[:, 0], q[:, 1], q[:, 2]]

        v00, v10, v11, v01 = corner(0, 0), corner(1, 0), corner(1, 1), corner(0, 1)

        # Occupied cell next to each face, to point the normal outwards.
        cell = idx.copy()
        cell[:, a] -= lo[tuple(idx.T)].astype(int)
        cell = np.clip(cell, 0, np.array(occ.shape) - 1)
        outward = (v00 + v11) / 2 - C[cell[:, 0], cell[:, 1], cell[:, 2]]
        normal = np.cross(v10 - v00, v01 - v00)
        flip = (normal * outward).sum(1) < 0

        t1 = np.stack([v00, v10, v11], 1)
        t2 = np.stack([v00, v11, v01], 1)
        t1[flip] = t1[flip][:, [0, 2, 1]]
        t2[flip] = t2[flip][:, [0, 2, 1]]
        tris += [t1, t2]
    return np.concatenate(tris)


def cone_piece(turns, th0, th1):
    """Mesh one piece of the cone strip, from angle th0 to th1."""
    width, _ = cone_dims(turns)
    nu = max(2, int(np.ceil((th1 - th0) * Cone.R_OUT / RES)))
    nv = max(2, int(np.ceil(width / RES)))
    th = np.linspace(th0, th1, nu + 1)
    fr = np.linspace(0, 1, nv + 1)
    TH, FR = np.meshgrid(th, fr, indexing="ij")
    P = np.stack([cone_points(turns, TH, FR, dz) for dz in (0, Cone.INLAY, Cone.THICKNESS)], axis=2)

    # Map value at each cell centre.
    THc, FRc = np.meshgrid((th[:-1] + th[1:]) / 2, (fr[:-1] + fr[1:]) / 2, indexing="ij")
    r = Cone.R_OUT - width * THc / (2 * np.pi) - FRc * width
    lon, lat = map_latlon(r, THc, Cone.R_OUT, -1)
    mask = world_at(lon, lat, smooth=False) > THRESHOLD
    if INVERT:
        mask = ~mask

    clear = np.zeros((nu, nv, 2), bool)
    clear[:, :, 0] = ~mask                   # bottom layer: clear where there is no map
    clear[:, :, 1] = True                    # top layer: always clear
    colour = np.zeros((nu, nv, 2), bool)
    colour[:, :, 0] = mask
    return voxel_mesh(P, clear), voxel_mesh(P, colour), (TH, FR, mask)


def cone_splice(turns, th_joint):
    """Thin plate glued on top of a joint, a bit narrower than the strip."""
    half = np.radians(Cone.SPLICE_DEG) / 2
    nu = max(2, int(np.ceil(2 * half * Cone.R_OUT / RES)))
    th = np.linspace(th_joint - half, th_joint + half, nu + 1)
    fr = np.linspace(0.1, 0.9, 3)
    TH, FR = np.meshgrid(th, fr, indexing="ij")
    P = np.stack([cone_points(turns, TH, FR, dz)
                  for dz in (Cone.THICKNESS, Cone.THICKNESS + Cone.SPLICE_T)], axis=2)
    return voxel_mesh(P, np.ones((nu, 2, 1), bool))

## Cone: pieces, splices and preview
Each piece is laid flat on the bed (lowest point at z = 0) and checked against the build volume. An assembled copy is also written, for viewing only.

In [ ]:
# Cell 8: cone generation

def fits_bed(size):
    """Check a piece against the build volume, allowing a 90° turn on the bed."""
    xy_fits = all(s <= b for s, b in zip(sorted(size[:2], reverse=True), sorted(BED[:2], reverse=True)))
    return xy_fits and size[2] <= BED[2]


def cone_preview(turns, previews):
    """Plot the map seen from below (red = joints) and a 3D side view."""
    width, _ = cone_dims(turns)
    fig = plt.figure(figsize=(14, 7))
    ax = fig.add_subplot(1, 2, 1)
    for TH, FR, mask in previews:
        r = Cone.R_OUT - width * TH / (2 * np.pi) - FR * width
        X, Y = r * np.cos(TH), r * np.sin(TH)
        ax.pcolormesh(X, Y, mask.astype(float), cmap="Greys", vmin=-0.3, vmax=1.3, shading="flat")
        ax.plot(X[[0, -1], :].T, Y[[0, -1], :].T, color="red", lw=1.5)
    ax.set_aspect("equal")
    ax.invert_xaxis()
    ax.axis("off")
    ax.set_title("underside, seen from below (red = joints)")

    ax3 = fig.add_subplot(1, 2, 2, projection="3d")
    colours = plt.cm.tab10(np.arange(len(previews)))
    for (TH, FR, _), colour in zip(previews, colours):
        step = max(1, TH.shape[0] // 60)
        P = cone_points(turns, TH[::step], FR[::step], 0)
        ax3.plot_surface(P[..., 0], P[..., 1], P[..., 2], color=colour, shade=True, linewidth=0)
    ax3.set_box_aspect((2 * Cone.R_OUT, 2 * Cone.R_OUT, Cone.HEIGHT))
    ax3.axis("off")
    ax3.set_title("side view (one colour per piece)")
    plt.tight_layout()
    plt.show()


def generate_cone(turns=Cone.TURNS):
    """Build all pieces, splices, the assembled view and the previews for the cone."""
    width, theta_total = cone_dims(turns)
    clear_stl_folder(Cone.OUT_DIR)
    print(f"{turns} turns: strip width {width:.1f} mm, drop {Cone.HEIGHT / turns:.0f} mm per turn")

    edges = np.linspace(0, theta_total, Cone.PIECES + 1)
    previews = []
    assembled = {"clear": [], "map": []}
    for p in range(Cone.PIECES):
        clear, colour, preview = cone_piece(turns, edges[p], edges[p + 1])
        previews.append(preview)
        assembled["clear"].append(clear)
        assembled["map"].append(colour)

        # Lay the piece on the bed: lowest point at z = 0, centred in x and y.
        allv = np.concatenate([clear, colour]).reshape(-1, 3)
        lo, hi = allv.min(0), allv.max(0)
        shift = np.array([-(lo[0] + hi[0]) / 2, -(lo[1] + hi[1]) / 2, -lo[2]])
        write_stl(f"{Cone.OUT_DIR}/piece_{p + 1}_clear.stl", clear + shift)
        write_stl(f"{Cone.OUT_DIR}/piece_{p + 1}_map.stl", colour + shift)
        size = hi - lo
        status = "fits" if fits_bed(size) else "!! DOES NOT FIT THE BED"
        print(f"  piece {p + 1}: {size[0]:.0f} x {size[1]:.0f} x {size[2]:.0f} mm, "
              f"{len(clear) + len(colour):,} triangles  {status}")

    for j in range(1, Cone.PIECES):
        splice = cone_splice(turns, edges[j])
        write_stl(f"{Cone.OUT_DIR}/splice_{j}-{j + 1}.stl", splice - splice.reshape(-1, 3).min(0))
        assembled["clear"].append(splice)

    # Whole spiral in place, for viewing only (not for printing).
    write_stl(f"{Cone.OUT_DIR}/assembled_clear.stl", np.concatenate(assembled["clear"]))
    write_stl(f"{Cone.OUT_DIR}/assembled_map.stl", np.concatenate(assembled["map"]))
    write_stl(f"{Cone.OUT_DIR}/assembled_full.stl", np.concatenate(assembled["clear"] + assembled["map"]))
    print("written to", Cone.OUT_DIR)
    cone_preview(turns, previews)

## Selector
Set `MODE` in Cell 1, pick the values and press **Run Interact**. This rebuilds all STLs and the previews.

In [ ]:
# Cell 9: interactive selector
import ipywidgets as widgets

if MODE == "flat":
    widgets.interact_manual(
        generate_flat,
        turns=widgets.IntSlider(value=Flat.TURNS, min=3, max=12, description="turns"),
        diameter=widgets.IntSlider(value=Flat.DIAMETER, min=100, max=1200, step=20,
                                   description="diameter mm"),
    )
else:
    widgets.interact_manual(
        generate_cone,
        turns=widgets.IntSlider(value=Cone.TURNS, min=2, max=8, description="turns"),
    )